# 实验十一 线性回归

线性回归用于预测连续数值，通过寻找一条直线最大程度地拟合样本特征与目标值之间的关系。

本实验先使用汽车价格与销量数据完成简单线性回归，再使用生成数据和加州房价数据，重点调整 `noise`、`test_size`、`fit_intercept` 和输入特征数量，观察拟合直线与评价指标变化。

## 1. 使用汽车价格与销量数据完成简单线性回归

计算汽车价格与销量数据的斜率和截距，绘制散点图及拟合直线，并预测新的销量。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

# 汽车价格（万元）与销量（千辆）
x = np.array([9.00, 9.23, 9.41, 9.68,
              9.98, 10.20, 10.58, 10.88,
              11.28, 11.56, 11.88, 12.00])
y = np.array([12.23, 11.70, 10.21, 9.60,
              8.72, 7.70, 7.10, 6.10,
              5.50, 4.65, 4.20, 3.50])

x_mean, y_mean = np.mean(x), np.mean(y)
a = ((x - x_mean).dot(y - y_mean) / (x - x_mean).dot(x - x_mean))
b = y_mean - a * x_mean
print('斜率 a =', a, '截距 b =', b)

y_hat = a * x + b

plt.scatter(x, y)
plt.plot(x, y_hat, color='r')
plt.xlabel('汽车价格（万元）')
plt.ylabel('汽车销量（千辆）')
plt.title('简单线性回归拟合')
plt.show()

> **操作要求**：记录斜率和截距，修改一个销量数据后重新运行，观察拟合直线和参数是否发生变化。

## 2. 修改数据噪声 noise

将 `noise` 设置为 5、20、50，比较拟合直线、MSE 和 R² 的变化。

In [ ]:
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

for noise_value in [5, 20, 50]:
    X, y = make_regression(n_samples=150, n_features=1, noise=noise_value, random_state=42)
    model = LinearRegression()
    model.fit(X, y)
    pred = model.predict(X)
    print('noise =', noise_value,
          'MSE =', round(mean_squared_error(y, pred), 2),
          'R² =', round(r2_score(y, pred), 4))

> **操作要求**：比较三组噪声下的评价指标，观察噪声增大后拟合效果的变化。

## 3. 修改训练集与测试集比例 test_size

将 `test_size` 设置为 0.1、0.2、0.3，观察测试样本数量和测试 R² 的变化。

In [ ]:
from sklearn.model_selection import train_test_split

X, y = make_regression(n_samples=200, n_features=3, noise=20, random_state=42)

for size in [0.1, 0.2, 0.3]:
    X_train, X_test, y_train, y_test = (
        train_test_split(X, y, test_size=size, random_state=42))
    model = LinearRegression()
    model.fit(X_train, y_train)
    print('test_size =', size,
          '测试样本数 =', len(X_test),
          'R² =', round(model.score(X_test, y_test), 4))

> **操作要求**：比较三种划分比例，记录测试样本数量和 R²，分析测试集过小时评价结果是否容易波动。

## 4. 调整截距参数 fit_intercept

比较 `fit_intercept=True` 与 `fit_intercept=False`，分析强制直线经过原点对预测结果的影响。

In [ ]:
for flag in [True, False]:
    model = LinearRegression(fit_intercept=flag)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print('fit_intercept =', flag,
          '截距 =', round(model.intercept_, 3),
          'MSE =', round(mean_squared_error(y_test, pred), 2))

> **操作要求**：比较是否保留截距时的 MSE 和 R²，观察强制模型经过原点是否会降低拟合效果。

## 5. 调整输入特征数量

分别使用 1 个、3 个和全部特征训练加州房价模型，比较 MSE、MAE 和 R²。

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import mean_absolute_error

housing = fetch_california_housing()
X, y = housing.data, housing.target

for n in [1, 3, X.shape[1]]:
    X_part = X[:, :n]
    X_train, X_test, y_train, y_test = (
        train_test_split(X_part, y, test_size=0.2, random_state=42))
    model = LinearRegression()
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print('特征数 =', n,
          'MAE =', round(mean_absolute_error(y_test, pred), 3),
          'R² =', round(model.score(X_test, y_test), 3))

> **操作要求**：比较不同特征数量下的评价指标，分析特征增多后模型预测能力的变化。

## 6. 创新拓展实践：噪声对回归结果的影响

在汽车价格与销量数据中加入不同程度的随机误差，比较原始数据和含噪声数据的回归结果。

任务：绘制三组散点图和拟合直线，记录斜率、截距和 R²，分析噪声的影响；可设置 `np.random.seed(42)` 保证结果可重复。

In [ ]:
np.random.seed(42)

x_car = np.array([9.0, 9.5, 10.0, 10.5, 11.0, 11.5, 12.0])
y_car = np.array([12.2, 10.5, 8.8, 7.1, 5.9, 4.6, 3.5])

from sklearn.metrics import r2_score

for scale in [0, 0.3, 0.8]:
    y_noise = y_car + np.random.normal(0, scale, size=len(y_car))
    model = LinearRegression().fit(x_car.reshape(-1, 1), y_noise)
    y_pred = model.predict(x_car.reshape(-1, 1))
    print('scale =', scale,
          '斜率 =', round(model.coef_[0], 3),
          '截距 =', round(model.intercept_, 3),
          'R² =', round(r2_score(y_noise, y_pred), 4))
    plt.scatter(x_car, y_noise)
    plt.plot(x_car, y_pred, color='r')
    plt.title(f'噪声 scale = {scale}')
    plt.show()